# Magnetic Resonance Imaging II
 
## Exercise 5
 
---
 
**Prof. Dr. Markus Zimmermann**
 
**Mohamed Wassim Bensalem**
 
---

> **Status note (work in progress):** this revision covers sequence assembly, the regridded reconstruction $\mathbf{A}^\mathrm{H}\mathbf{d}$, and CG reconstruction only (Tasks 1–4 below). The comparison against Total Variation / ADMM (as in Exercise 4) and a self-supervised learning (SSL) reconstruction will be added in a later revision — do not implement those yet.
 
In Exercise 3 (EPI) a single excitation traversed all of k-space along a Cartesian zig-zag; in Exercise 4 (radial) each excitation acquired one straight spoke through the k-space center. This exercise introduces **spiral imaging**: after a single excitation, the readout gradients trace a spiral outward from the center of k-space to its edge, covering a 2D disc rather than a line. Spiral trajectories are attractive because they are very gradient-efficient (no wasted time on phase-encoding blips or line-to-line jumps), start sampling at $k=0$ where the signal is strongest, and — like radial — degrade gracefully under undersampling rather than producing coherent Cartesian aliasing.
 
The cost is reconstruction complexity: a spiral trajectory is not just off-grid like radial, it is also non-uniform in **density** — samples are packed close together near the center and spread apart near the edge — and each interleaf/shot is a different rotated copy of the same spiral arm. As in Exercise 4, this means the standard FFT cannot be used and the **Non-Uniform Fourier Transform (NUFFT)** is required, together with the same `NonuniformFourierTransformOperator` you built there.
 
As with radial, an **undersampling factor `R`** controls how many spiral interleaves are actually acquired relative to the number required for full Nyquist sampling. Fewer interleaves means faster scans but larger gaps between the spiral arms, and therefore more pronounced streaking-type artifacts — you will see this directly by varying `R`.

## Software Imports
 
All required libraries and modules are imported here.

In [ ]:
%matplotlib widget

import os
import sys
from typing import Optional, Tuple

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
import pypulseq as pp
import torch
from tqdm import tqdm

sys.path.insert(0, "/home/jovyan/mri/src")
sys.path.insert(0, "/home/jovyan/shared-folder/Magnetresonanztomographie-2/mri/src")

from mri.conopt.functional.fourier import (
    nonuniform_fourier_transform_adjoint,
    nonuniform_fourier_transform_forward,
)
from mri.conopt.linops import LinearOperator
from mri.parim.analytic import cyclic_head_coil

## System and Sequence Parameters
 
`n_interleaves_full` is the number of spiral interleaves required for full Nyquist sampling of k-space (design quantity, fixed regardless of `R`). `R` then controls how many of those interleaves are actually **acquired**: `n_interleaves_acquired = n_interleaves_full // R`. Start with `R = 1` to verify the pipeline on a fully-sampled spiral, then increase `R` to see the effect of undersampling on the spiral's arm spacing.
 
`us_edge` and `alpha_taper` control **variable density**: `us_edge = 1.0` gives a standard constant-density spiral; `us_edge > 1.0` deliberately under-samples the outer part of k-space (even at `R = 1`) to shorten the readout, trading high-spatial-frequency SNR for speed.

In [ ]:
phantom_path_local = "/home/jovyan/mri-data/mri-datasets/brainweb"
phantom_path_shared = "/home/jovyan/shared-folder/Magnetresonanztomographie-2/mri-data/mri-datasets/brainweb"
seq_filename = "spiral.seq"

# fov_x, fov_y = 256e-3, 256e-3
fov_x, fov_y = 256e-3, 256e-3
n_x, n_y = 128, 128
n_channels = 8

n_interleaves_full = 16  # design value; satisfies Nyquist sampling at R = 1
R = 1  # undersampling factor: acquire n_interleaves_full // R shots

us_edge = 4.0  # 1.0 = constant density; > 1.0 = variable density (undersample edge)
alpha_taper = 1.0  # shape of the density taper

case = ["T1-weighted", "T2-weighted", "PD-weighted"][2]

if case == "T1-weighted":
    flip_angle_deg = 90
    tr = 250e-3
    te = 5e-3
elif case == "T2-weighted":
    flip_angle_deg = 40
    tr = 5000e-3
    te = 50e-3
elif case == "PD-weighted":
    flip_angle_deg = 40
    tr = 5000e-3
    te = 5e-3

num_dummy_pulses = 50

adc_oversample = 1  # ADC samples this many times finer than the gradient raster;
# lets the along-track (within-interleaf) Nyquist requirement be met
# via ADC bandwidth rather than by slowing the trajectory down

torch.set_default_device("cpu")

## System and Sequence Object

In [ ]:
system = pp.Opts(
    max_grad=28,
    grad_unit="mT/m",
    max_slew=150,
    slew_unit="T/m/s",
)

seq = pp.Sequence(system)

## Linear Operators

`ChannelOperator` and `NonuniformFourierTransformOperator` are identical to Exercise 4 and are provided here so you can focus on the spiral-specific parts of this exercise.

In [ ]:
class ChannelOperator(LinearOperator):

    def __init__(
        self,
        coil_sensitivities: torch.Tensor,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        nC = coil_sensitivities.shape[0]

        self.coil_sensitivities = coil_sensitivities
        self.device = device
        self.dtype = torch.float32
        self.internal_dtype = torch.complex64
        self.forward_shape = (1, *data_shape)
        self.adjoint_shape = (nC, *data_shape)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(self.internal_dtype).reshape(self.forward_shape)
        y = self.coil_sensitivities * x
        return y.view(torch.float32).ravel()

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(self.internal_dtype).reshape(self.adjoint_shape)
        y = torch.sum(torch.conj(self.coil_sensitivities) * x, dim=0, keepdim=True)
        return y.view(torch.float32).ravel()


class NonuniformFourierTransformOperator(LinearOperator):

    def __init__(
        self,
        k: torch.Tensor,
        input_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        nD, nK = k.shape
        nC, nX, nY = input_shape
        self.n_modes = (nX, nY)

        self.k = k
        self.forward_shape = (nC, nX, nY, 1)
        self.adjoint_shape = (nC, nK)

        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )

        self.dtype = torch.float32
        self.internal_dtype = torch.complex64
        self.device = device

    def _matvec(self, input_tensor: torch.Tensor) -> torch.Tensor:
        input_tensor = input_tensor.view(self.internal_dtype).reshape(
            self.forward_shape
        )
        output_tensor = nonuniform_fourier_transform_forward(
            self.k,
            input_tensor,
        )
        return output_tensor.ravel().view(self.dtype)

    def _rmatvec(self, input_tensor: torch.Tensor) -> torch.Tensor:
        input_tensor = input_tensor.view(self.internal_dtype).reshape(
            self.adjoint_shape
        )
        output_tensor = nonuniform_fourier_transform_adjoint(
            self.k,
            input_tensor,
            self.n_modes,
        )
        return output_tensor.ravel().view(self.dtype)

## Numerical Solvers

The CG solver is identical to Exercises 2–4 and is provided here.

In [ ]:
def conjugate_gradient(
    Q: LinearOperator,
    b: torch.Tensor,
    x: torch.Tensor,
    maxiter: int = 10,
    disable_progressbar: bool = False,
) -> torch.Tensor:

    g_prev = torch.ones(x.shape, dtype=x.dtype)
    p_prev = torch.zeros(x.shape, dtype=x.dtype)

    for iteration in tqdm(
        range(maxiter),
        disable=disable_progressbar,
    ):

        g = Q @ x - b

        conjugacy_factor = g @ g / (g_prev @ g_prev)
        p = g + conjugacy_factor * p_prev
        stepsize = g @ g / (p @ (Q @ p))
        x = x - stepsize * p

        g_prev = g.clone()
        p_prev = p.clone()

    return x

## Sequence Components
 
The spiral is designed in two stages, both provided here so you can focus on turning the result into a PyPulseq sequence:
 
**1. Trajectory shape.** At every k-space radius $r$ we choose a local field-of-view $\text{FOV}(r)$ (constant for `us_edge = 1`, shrinking toward the edge for `us_edge > 1`). The sampling-density condition for an `n_interleaves_full`-shot spiral is
 
$$\frac{d\theta}{dr} = \frac{2\pi \cdot \text{FOV}(r)}{n_\text{interleaves,full}}$$
 
Integrating this from $r=0$ to $r=k_\text{max}$ gives $\theta(r)$ and hence the k-space path of a single interleaf, $k(r) = r \cdot e^{i\theta(r)}$. This path shape does **not** depend on `R` — undersampling is applied later, purely by rotating fewer copies of it (see Task 1).
 
**2. Time parametrization.** The path above says nothing about *how fast* to traverse it — that is set by the gradient amplitude and slew-rate limits, **and** by one more constraint that is easy to miss: the ADC's dwell time. Exactly as in the EPI and radial exercises, where the readout gradient amplitude was chosen so that `Δk` per sample equals `1/FOV`, the spiral's local k-space *speed* `|dk/dt|` must never exceed `1/(FOV · dwell)` — otherwise consecutive ADC samples along the fast, outer part of the spiral end up farther apart than Nyquist allows, under-sampling exactly the highest spatial frequencies (i.e. the sharpest edges in the image, such as the skull boundary). This is independent of, and in addition to, the two constraints already discussed below. Because the spiral is curved, the slew-rate budget must also be split between the *tangential* component (speeding up/slowing down) and the *centripetal* component (changing direction, proportional to $v^2 \kappa$ where $\kappa$ is the local path curvature). A forward/backward speed-limiting pass — the non-Cartesian analogue of a trapezoid ramp calculation — combines all three limits (hardware amplitude, hardware+curvature slew, and FOV/dwell) into a curvature- and bandwidth-aware time axis, which is then used to resample the path onto the gradient raster and differentiate it into gradient waveforms `gx`, `gy` (units Hz/m, directly compatible with `pp.make_arbitrary_grad`).
 
Note the FOV/dwell constraint above uses `dwell`, not the gradient raster — real ADC hardware can sample much faster than the gradient DAC updates, so the ADC is deliberately given a finer dwell (`grad_raster / adc_oversample`) than the 10 µs used for the gradient waveform itself. This is what keeps the readout short: capping speed at the *coarse* raster's dwell would force the whole trajectory to slow down to stay Nyquist-compliant (an unnecessarily long readout), whereas capping it at the *fine* ADC dwell lets the trajectory run close to the hardware amplitude/slew limit — exactly like the original design — while still resolving enough along-track points to satisfy Nyquist. The gradient waveform (`gx`, `gy`, handed to `pp.make_arbitrary_grad`) still lives on the coarse 10 µs raster required by gradient hardware; only the ADC's own sampling times are finer. PyPulseq's `calculate_kspace()` correctly resolves the trajectory at these finer ADC times by interpolating the (piecewise-linear) gradient waveform between raster points, so no information is lost or invented in between.
 
Run this cell as-is.


In [ ]:
delta_kx = 1 / fov_x
fov = fov_x
kmax = n_x / (2 * fov)

grad_raster = system.grad_raster_time
adc_dwell = grad_raster / adc_oversample
Gmax = 0.95 * system.max_grad
Smax = 0.90 * system.max_slew

In [ ]:
def local_fov(r_norm):
    """Variable-density FOV(r); r_norm = r / kmax in [0, 1]."""
    return fov * (1 - (1 - 1 / us_edge) * r_norm**alpha_taper)


# --- shape: integrate dtheta/dr ---
n_shape = 20000
r = np.linspace(1e-6, kmax, n_shape)
dtheta_dr = 2 * np.pi * local_fov(r / kmax) / n_interleaves_full
theta = np.concatenate(
    ([0.0], np.cumsum(0.5 * (dtheta_dr[1:] + dtheta_dr[:-1]) * np.diff(r)))
)
k_shape = r * np.exp(1j * theta)
k_shape = np.concatenate(([0.0 + 0j], k_shape))  # start exactly at k = 0

# --- timing: curvature- and slew-limited forward/backward speed pass ---
ds = np.abs(np.diff(k_shape))
N = len(k_shape)

x_path, y_path = k_shape.real, k_shape.imag
dx, dy = np.gradient(x_path), np.gradient(y_path)
d2x, d2y = np.gradient(dx), np.gradient(dy)
denom = (dx**2 + dy**2) ** 1.5
kappa = np.abs(dx * d2y - dy * d2x) / np.maximum(denom, 1e-30)

a_split = Smax / np.sqrt(2)  # split slew budget: tangential / centripetal
v_curve_max = np.sqrt(a_split / np.maximum(kappa, 1e-9))
v_readout_max = 1 / (
    fov * adc_dwell
)  # FOV/dwell Nyquist limit, using the FINE adc dwell
# (not the coarse gradient raster) so the trajectory
# isn't forced to slow down unnecessarily
v_cap = np.minimum(np.minimum(Gmax, v_curve_max), v_readout_max)

v_f = np.zeros(N)
v_b = np.zeros(N)
for i in range(1, N):
    v_f[i] = min(v_cap[i], np.sqrt(v_f[i - 1] ** 2 + 2 * a_split * ds[i - 1]))
for i in range(N - 2, -1, -1):
    v_b[i] = min(v_cap[i], np.sqrt(v_b[i + 1] ** 2 + 2 * a_split * ds[i]))
v = np.maximum(np.minimum(v_f, v_b), 1e-6)

dt = ds / (0.5 * (v[1:] + v[:-1]))
t_path = np.concatenate(([0.0], np.cumsum(dt)))

t_grid = np.arange(0, t_path[-1], grad_raster)
kx_grid = np.interp(t_grid, t_path, k_shape.real)
ky_grid = np.interp(t_grid, t_path, k_shape.imag)

In [ ]:
# --- differentiate: base gradient waveform of a single interleaf, [Hz/m] ---
# This stays on the coarse 10 us gradient raster, as gradient hardware requires.
gx = np.diff(kx_grid) / grad_raster
gy = np.diff(ky_grid) / grad_raster
gx = np.concatenate(([0.0], gx))
gy = np.concatenate(([0.0], gy))

# The ADC samples the SAME designed-spiral duration (t_path[-1]) but at its
# own, finer dwell time -- this is what lets the trajectory move at close to
# the hardware amplitude/slew limit (short readout) while still resolving
# enough along-track points to satisfy Nyquist (see note above). It does NOT
# sample the ramp-down below: n_adc is fixed at this point, before the ramp
# is appended to gx/gy.
# n_adc = int(np.floor(t_path[-1] / adc_dwell))
n_adc = len(kx_grid)

# ramp both gradient channels back to zero at the end of the readout
# (gradient waveform only -- the ADC above already stopped sampling)
ramp_pts = (
    int(np.ceil(max(np.max(np.abs(gx)), np.max(np.abs(gy))) / Smax / grad_raster)) + 1
)
ramp = np.linspace(1, 0, ramp_pts)
gx = np.concatenate((gx, gx[-1] * ramp))
gy = np.concatenate((gy, gy[-1] * ramp))

print(
    f"Spiral readout: {n_adc} sampled points/interleaf ({n_adc * adc_dwell * 1e3:.2f} ms, "
    f"adc_dwell={adc_dwell * 1e6:.2f} us), "
    f"+{ramp_pts} unsampled ramp-down points (on the {grad_raster * 1e6:.0f} us gradient raster), "
    f"n_interleaves_full={n_interleaves_full}, R={R}, "
    f"n_interleaves_acquired={n_interleaves_full // R}"
)

assert np.max(np.abs(gx)) <= system.max_grad + 1, "Gx exceeds max_grad"
assert np.max(np.abs(gy)) <= system.max_grad + 1, "Gy exceeds max_grad"

plt.figure()
plt.plot(kx_grid, ky_grid, lw=0.7)
plt.gca().set_aspect("equal")
plt.xlabel("kx [1/m]")
plt.ylabel("ky [1/m]")
plt.title("Base spiral interleaf (before rotation)")
plt.show()

## Task 1: Spiral Sequence Assembly

`gx` and `gy` above describe a single spiral interleaf. To build the full sequence, every acquired interleaf is a **rotated copy** of this same base waveform:
 
$$g_{x,i} = \cos(\phi_i)\, g_x - \sin(\phi_i)\, g_y, \qquad g_{y,i} = \sin(\phi_i)\, g_x + \cos(\phi_i)\, g_y$$
 
with rotation angle $\phi_i = 2\pi i / n_\text{interleaves,acquired}$ for $i = 0, \dots, n_\text{interleaves,acquired} - 1$. Note that the rotation angles always span the full circle $[0, 2\pi)$ regardless of `R` — undersampling comes purely from spreading **fewer** interleaves over that same full circle, leaving larger angular gaps between arms (exactly analogous to how fewer radial spokes were spread over $[0,\pi)$ in Exercise 4).
 
As in Exercise 4, `num_dummy_pulses` preparatory TRs are played first (ADC switched off, angle fixed at $\phi=0$) to reach steady state before acquisition begins.
 
**Your tasks:**
 
1. Compute `n_interleaves_acquired = n_interleaves_full // R`.
2. Define `rf` (non-selective — no slice gradient needed), `adc` (`num_samples=n_adc`, `dwell=grad_raster`), and a spoiler pair `gx_spoiler`/`gy_spoiler`.
3. Loop over `index in range(-num_dummy_pulses, n_interleaves_acquired)`. For each iteration: pick the rotation angle (`0` for dummy pulses, `2*pi*index/n_interleaves_acquired` otherwise), rotate `gx`/`gy`, wrap the rotated waveforms with `pp.make_arbitrary_grad`, and assemble the block (RF, then gradients with or without `adc`, then spoiler, then a TR delay).

In [ ]:
# Your solution here


rf = pp.make_block_pulse(
    flip_angle=np.deg2rad(flip_angle_deg),
    duration=0.5e-3,
    system=system,
)

adc = pp.make_adc(
    num_samples=n_adc,
    dwell=grad_raster,
    delay=system.adc_dead_time,
    system=system,
)

gx_spoiler = pp.make_trapezoid(
    channel="x",
    area=4 * kmax,
    system=system,
)

gy_spoiler = pp.make_trapezoid(
    channel="y",
    area=4 * kmax,
    system=system,
)

In [ ]:
# Your solution here


n_interleaves_acquired = n_interleaves_full // R

for index in range(-num_dummy_pulses, n_interleaves_acquired):

    phi = 2 * np.pi * index / n_interleaves_acquired

    gx_i = gx * np.cos(phi) - gy * np.sin(phi)
    gy_i = gx * np.sin(phi) + gy * np.cos(phi)

    gx_shot = pp.make_arbitrary_grad(
        channel="x",
        waveform=gx_i,
        system=system,
    )
    gy_shot = pp.make_arbitrary_grad(
        channel="y",
        waveform=gy_i,
        system=system,
    )

    seq.add_block(rf)

    if index < 0:
        seq.add_block(gx_shot, gy_shot)
    else:
        seq.add_block(gx_shot, gy_shot, adc)

    seq.add_block(gx_spoiler, gy_spoiler)

    t_used = (
        pp.calc_duration(rf)
        + pp.calc_duration(gx_shot, gy_shot, adc)
        + pp.calc_duration(gx_spoiler, gy_spoiler)
    )

    if tr > t_used:
        print(tr, t_used)
        seq.add_block(pp.make_delay(tr - t_used))

## Verification and Export

In [ ]:
ok, error_report = seq.check_timing()

if ok:
    print("Timing check passed successfully")
else:
    print("Timing check failed. Error listing follows:")
    [print(e) for e in error_report]

plt.figure()
seq.plot(time_range=(0.0, n_interleaves_acquired * tr))

seq.write(seq_filename, v141_compat=True)

## Phantom and Multi-Channel Simulation

As in Exercise 3, this uses the realistic Brainweb digital brain phantom. Run this cell as-is.

In [ ]:
phantom = mr0.util.load_phantom(size=(n_x, n_y))

coil_sensitivities_recon = cyclic_head_coil(
    (n_channels, n_x, n_y, 1),
).to(torch.complex64)
coil_sensitivities_recon = coil_sensitivities_recon.flip((1, 2))
coil_sensitivities_recon = coil_sensitivities_recon[..., 0]

coil_sensitivities_sim = cyclic_head_coil(
    (n_channels, n_x, n_y, 1),
    fov_scaling=(phantom.size[0] / fov_x, phantom.size[1] / fov_y, 1),
).to(torch.complex64)
phantom.coil_sens = coil_sensitivities_sim

# n_channels = 1
# phantom.coil_sens = torch.ones((n_channels, n_x, n_y, 1), dtype=torch.complex64)
# coil_sensitivities_recon = torch.ones((n_channels, n_x, n_y), dtype=torch.complex64)

plt.figure()
phantom.plot()

mr0_seq = mr0.Sequence.import_file(seq_filename)

data = phantom.build()
graph = mr0.compute_graph(mr0_seq, data)
signal = mr0.execute_graph(graph, mr0_seq, data, print_progress=True)

## Task 2: Coil Images
 
As in Exercise 4, reconstruct each coil image with the NUFFT adjoint rather than an FFT, since the spiral samples do not lie on a Cartesian grid. The k-space trajectory is again read from `seq.calculate_kspace()`; only the acquired samples matter (dummy-pulse TRs carry no ADC and therefore contribute no k-space rows).
 
**Your tasks:**
 
1. Reshape and permute `signal` to `data` of shape $(n_\text{channels}, n_K)$, then call `.contiguous()`.
2. Extract `k_traj_adc` from `seq.calculate_kspace()`, keep only $k_x, k_y$, normalize to $[-0.5, 0.5]$, and convert to `torch.float32`.
3. Call `nonuniform_fourier_transform_adjoint` and drop the trailing dimension to get `coil_images`.
4. Display the coil images and coil sensitivity maps in 2×4 grids.

In [ ]:
# Your solution here


data = signal.clone()
data = data.reshape((-1, n_channels)).permute((1, 0))
data = data.contiguous()

kspace = seq.calculate_kspace()
k_traj_adc = kspace[0]
k_traj_adc = k_traj_adc[:2, :]
k_traj_radius = np.sqrt(np.sum(k_traj_adc**2, axis=0, keepdims=True))
k_traj_adc = k_traj_adc / k_traj_radius.max() * 0.5
k_traj_adc = torch.tensor(k_traj_adc, dtype=torch.float32)

coil_images = nonuniform_fourier_transform_adjoint(k_traj_adc, data, (n_x, n_y))[..., 0]

plt.figure()
for i in range(n_channels):
    plt.subplot(2, 4, i + 1)
    plt.imshow(torch.abs(coil_images[i, :, :]).numpy().T, cmap="gray")
    plt.title(f"Channel {i + 1}")
plt.suptitle("Coil images")
plt.tight_layout()
plt.show()

plt.figure()
for i in range(n_channels):
    plt.subplot(2, 4, i + 1)
    plt.imshow(torch.abs(coil_sensitivities_recon[i, :, :]).numpy().T, cmap="viridis")
    plt.title(f"Channel {i + 1}")
plt.suptitle("Coil sensitivity maps")
plt.tight_layout()
plt.show()

## Task 3: Compose the Forward Operator and Compute $\mathbf{A}^\mathrm{H} \mathbf{d}$
 
The forward operator is $\mathbf{A} = \mathbf{F}_\text{nu}\, \mathbf{C}$, exactly as in Exercise 4 — there is no sampling mask, since the trajectory `k_traj_adc` already encodes which (non-Cartesian) locations were sampled.
 
**Your tasks:**
 
1. Instantiate `F` as a `NonuniformFourierTransformOperator` with `k_traj_adc` and `input_shape=(n_channels, n_x, n_y)`.
2. Instantiate `C` as a `ChannelOperator` with `coil_sensitivities_recon` and `data_shape=(n_x, n_y)`.
3. Compose `A = F @ C`.
4. Compute `AHd = A.H @ data.view(torch.float32).ravel()`.
5. Reshape and display the magnitude of `AHd`.

In [ ]:
# Your solution here


F = NonuniformFourierTransformOperator(k_traj_adc, (n_channels, n_x, n_y))
C = ChannelOperator(coil_sensitivities_recon, (n_x, n_y))

A = F @ C

AHd = A.H @ data.view(torch.float32).ravel()

AHd_image = AHd.view(torch.complex64).reshape((n_x, n_y))

plt.figure()
plt.imshow(torch.abs(AHd_image).numpy().T, cmap="gray")
plt.title(
    f"$\\mathbf{{A}}^\\mathrm{{H}} \\mathbf{{d}}$ — Regridded Reconstruction (R={R})"
)
plt.show()

## Task 4: Iterative Reconstruction with Conjugate Gradient
 
Run CG to solve $\mathbf{Q}\mathbf{x} = \mathbf{b}$ where $\mathbf{Q} = \mathbf{A}^\mathrm{H}\mathbf{A}$ and $\mathbf{b} = \mathbf{A}^\mathrm{H}\mathbf{d}$.
 
**Your tasks:**
 
1. Construct $\mathbf{Q} = \mathbf{A}^\mathrm{H}\mathbf{A}$ and set $\mathbf{b} = \mathbf{A}^\mathrm{H}\mathbf{d}$.
2. Initialize $\mathbf{x}$ as a zero tensor of shape $(2 \cdot n_x \cdot n_y,)$ with `dtype=torch.float32`.
3. Run `conjugate_gradient(Q, b, x, maxiter=30)` and display the result alongside the zero-filled/regridded image.

In [ ]:
# Your solution here


Q = A.H @ A

b = AHd
x = torch.zeros(n_x * n_y, dtype=torch.complex64).view(torch.float32)

x_recon = conjugate_gradient(Q, b, x, maxiter=50)

x_recon_image = x_recon.view(torch.complex64).reshape((n_x, n_y))

plt.figure()
plt.subplot(1, 2, 1)
plt.imshow(torch.abs(AHd_image).numpy().T, cmap="gray")
plt.title(f"Regridded (R={R})")
plt.subplot(1, 2, 2)
plt.imshow(torch.abs(x_recon_image).numpy().T, cmap="gray")
plt.title(f"CG Reconstruction (R={R})")
plt.tight_layout()
plt.show()

**To be continued:** try `R = 1, 2, 4` and observe how the spiral arm spacing and CG reconstruction quality change — the same experiment as Exercise 4's radial `R` sweep, but now for spiral undersampling. The next revision of this notebook will add a Total Variation / ADMM reconstruction (Exercise 4, Task 7–8 style) and a self-supervised learning (SSL) reconstruction for comparison against CG at higher `R`.

In [ ]:
image = phantom.PD * torch.exp(-te / phantom.T2)
image = image.to(torch.complex64).flip((1,))

In [ ]:
plt.figure()
plt.plot(k_traj_adc[0, :], k_traj_adc[1, :], ".")
plt.show()

In [ ]:
coil_images = coil_sensitivities_sim.flip((1, 2)) * image[None, ...]
kdata = nonuniform_fourier_transform_forward(k_traj_adc, coil_images)

In [ ]:
plt.figure()
plt.imshow(torch.abs(coil_images[0, ..., 0]).numpy().T, cmap="gray")
plt.title(
    f"$\\mathbf{{A}}^\\mathrm{{H}} \\mathbf{{d}}$ — Regridded Reconstruction (R={R})"
)
plt.show()

In [ ]:
AHd = A.H @ kdata.view(torch.float32).ravel()

AHd_image = AHd.view(torch.complex64).reshape((n_x, n_y))

plt.figure()
plt.imshow(torch.abs(AHd_image).numpy().T, cmap="gray")
plt.title(
    f"$\\mathbf{{A}}^\\mathrm{{H}} \\mathbf{{d}}$ — Regridded Reconstruction (R={R})"
)
plt.show()

In [ ]:
# Your solution here


Q = A.H @ A

b = AHd
x = torch.zeros(n_x * n_y, dtype=torch.complex64).view(torch.float32)

x_recon = conjugate_gradient(Q, b, x, maxiter=50)

x_recon_image = x_recon.view(torch.complex64).reshape((n_x, n_y))

plt.figure()
plt.subplot(1, 2, 1)
plt.imshow(torch.abs(AHd_image).numpy().T, cmap="gray")
plt.title(f"Regridded (R={R})")
plt.subplot(1, 2, 2)
plt.imshow(torch.abs(x_recon_image).numpy().T, cmap="gray")
plt.title(f"CG Reconstruction (R={R})")
plt.tight_layout()
plt.show()